In [1]:
import numpy as np
from astropy.nddata import block_replicate, block_reduce

In [ ]:
def upscale(
    data: np.array,
    upscale_y: int = 1,
    upscale_x: int = 1,
) -> np.array:
    """
    Upscaling.
    """
    if not (
        (isinstance(upscale_y, int) and upscale_y > 0) and
        (isinstance(upscale_x, int) and upscale_x > 0)
    ):
        raise ValueError("Upscaling factors must be positive integers.")
    
    for i, f in enumerate((upscale_y, upscale_x)):
        data = np.repeat(data, f, axis=i)
    
    return data/np.prod((upscale_y, upscale_x))

In [48]:
data = np.ones((400, 400))

fy, fx = 2, 4

block_reduce(data, (fy, fx), func=np.sum).shape

(200, 100)

In [ ]:
def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale.
    """
    n, m = data.shape
    u, v = n // downscale_y, m // downscale_x

    reshaped = np.zeros((u, v))
    




data = np.random.uniform(0, 10, (100, 100))


fy, fx = 1, 4
sampled_data = downscale(data, fy, fx)

ValueError: Upscaling factors must be positive integers.

In [31]:
np.abs(data.sum() - sampled_data.sum()) < 1e-16

np.True_

In [32]:
sampled_data.shape

(100, 400)

In [2]:
d = np.arange(81).reshape((9, 9)); d

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
       [ 9, 10, 11, 12, 13, 14, 15, 16, 17],
       [18, 19, 20, 21, 22, 23, 24, 25, 26],
       [27, 28, 29, 30, 31, 32, 33, 34, 35],
       [36, 37, 38, 39, 40, 41, 42, 43, 44],
       [45, 46, 47, 48, 49, 50, 51, 52, 53],
       [54, 55, 56, 57, 58, 59, 60, 61, 62],
       [63, 64, 65, 66, 67, 68, 69, 70, 71],
       [72, 73, 74, 75, 76, 77, 78, 79, 80]])

In [8]:
n, m = d.shape

fy, fx = 3, 3

u, v = n // fy, m // fx


d.reshape(u, v, fy, fx)#.sum(axis=(2, 3))

array([[[[ 0,  1,  2],
         [ 3,  4,  5],
         [ 6,  7,  8]],

        [[ 9, 10, 11],
         [12, 13, 14],
         [15, 16, 17]],

        [[18, 19, 20],
         [21, 22, 23],
         [24, 25, 26]]],


       [[[27, 28, 29],
         [30, 31, 32],
         [33, 34, 35]],

        [[36, 37, 38],
         [39, 40, 41],
         [42, 43, 44]],

        [[45, 46, 47],
         [48, 49, 50],
         [51, 52, 53]]],


       [[[54, 55, 56],
         [57, 58, 59],
         [60, 61, 62]],

        [[63, 64, 65],
         [66, 67, 68],
         [69, 70, 71]],

        [[72, 73, 74],
         [75, 76, 77],
         [78, 79, 80]]]])

In [64]:
data = np.arange(400*500).reshape((400, 500))

f_factor = np.array((4, 5))

nblocks = np.array(data.shape) // f_factor
new_shape = tuple(k for ij in zip(nblocks, f_factor) for k in ij)

new_shape

(np.int64(100), np.int64(4), np.int64(100), np.int64(5))

In [65]:
even_blocks_idx = tuple(range(0, len(new_shape), 2))
odd_block_idx = tuple(range(1, len(new_shape), 2))

reshaped = data.reshape(new_shape).transpose(even_blocks_idx + odd_block_idx)

In [66]:
reshaped = reshaped.sum(axis=(2, 3))

In [67]:
tuple(range(0, len(new_shape), 2)) + tuple(range(1, len(new_shape), 2))

(0, 2, 1, 3)

In [ ]:
def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale.
    """
    def handle_blocks_div(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """
        Handles blocks division. Arrays with shape not evenly divisible
        by the downscaling factor are cut to prevent errors.

        # TODO: insert effective sum conservation
        """
        nblocks = np.array(data.shape) // downscaling
        cut_shape = nblocks * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != cut_shape[ax]:
                data = data.swapaxes(0, ax)
                data = data[:cut_shape[ax]]
                data = data.swapaxes(0, ax)
        
        if np.any(np.mod(data.shape, cut_shape) != 0):
            raise ValueError("aaa")
        
        return data

    downscaling = np.array((downscale_y, downscale_x))
    data = handle_blocks_div(data, downscaling)
    nblocks = np.array(data.shape) // downscaling

    blocks_shape = tuple(dim for dims in zip(nblocks, f_factor) for dim in dims)
    downsampled_data = data.reshape(blocks_shape).transpose((0, 2, 1, 3)).sum(axis=(2, 3))
    return downsampled_data





def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale.
    """
    def _handle_shape(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Adjusts input array to be subdivided into blocks."""
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = data[:adj_shape[ax]]
                data = data.swapaxes(0, ax)
        return data


    def _to_blocks(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Reshapes input array into blocks."""
        pass

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data_blocks = _to_blocks(data, downscaling)
    return data_blocks.sum(axis=(2, 3))










a = np.ones((10, 10))
fy, fx = 5, 5
reduced_a = block_reduce(a, (fy, fx))
downsampled_a = downscale(a, *(fy, fx))


np.all(downsampled_a == reduced_a)

ValueError: cannot reshape array of size 100 into shape (2,4,2,5)

In [81]:
np.sum(a), np.sum(downsampled_a)

(np.float64(121.0), np.float64(80.0))